### What does overfitting in CNNs look like in the Hessian eigenspectrum?

This notebook will test convolutional neural networks trained on the MNIST dataset for overfitting, and exploring how this looks in the eigenspectrum of the Hessian matrix.

In [1]:
# Libraries import

from multiprocessing import freeze_support

import os
import sys
import pickle
import pprint
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

sys.path.append("../")

import torch as t
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

from tqdm import tqdm
from datetime import datetime
import json
import wandb

from devinterp.slt import estimate_learning_coeff, estimate_learning_coeff_with_summary
from devinterp.optim.sgld import SGLD
from devinterp.slt import sample
from devinterp.slt.llc import OnlineLLCEstimator
from devinterp.slt.wbic import OnlineWBICEstimator

from approxngd import KFAC
from PyHessian.pyhessian import *
from PyHessian.density_plot import *
from general_utils import *
from hessian_utils import *
from networks import *
from data.build_data import build_data

import plotly.express as px
import plotly.graph_objects as go
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

device = "cuda" if t.cuda.is_available() else "cpu"
print(device)

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
hyperparams = {
    "batch_size" : 64,
    "num_workers" : 12,
    "num_epochs" : 10,
    "lr" : 9e-03,
    "momentum" : 0.9,
}

In [3]:
# Produce a list of CNNs

hidden_conv_layers = [8]
models = {}
for hidden_conv_layer in hidden_conv_layers:
    models[hidden_conv_layer] = CnnMNIST(hidden_conv_layers=hidden_conv_layer).to(device)

In [4]:
# load data from pytorch dataloaders

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = t.utils.data.DataLoader(train_set, batch_size=hyperparams["batch_size"], shuffle=True, num_workers=hyperparams["num_workers"], persistent_workers=True)
test_loader = t.utils.data.DataLoader(test_set, batch_size=hyperparams["batch_size"], shuffle=False, num_workers=hyperparams["num_workers"], persistent_workers=True)

In [5]:
# training loop (vary number of hidden layers in CNN)

train_losses = {}
test_losses = {}
for hidden_conv_layer, model in models.items():
    print(f"TRAINING WITH {hidden_conv_layer} CONVOLUTION LAYERS")
    train_losses[hidden_conv_layer] = []
    test_losses[hidden_conv_layer] = []
    sgd = t.optim.SGD(model.parameters(), lr=hyperparams["lr"], momentum=hyperparams["momentum"], nesterov=True)
    ngd = KFAC(model, hyperparams["lr"], damping=0.01, momentum_type='regular', momentum=0.8, adapt_damping=False, update_cov_manually=False)
    optimiser = sgd
    criterion = nn.CrossEntropyLoss(reduction='mean') if optimiser==ngd else nn.CrossEntropyLoss()
    for epoch in range(1, hyperparams["num_epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimiser, criterion, device)
        test_loss = evaluate(model, test_loader, criterion, device)
        train_losses[hidden_conv_layer].append(train_loss)
        test_losses[hidden_conv_layer].append(test_loss)
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}: train_loss={train_loss:.4f}, test_loss={test_loss:.4f}")

TRAINING WITH 8 CONVOLUTION LAYERS


  0%|          | 0/938 [00:00<?, ?it/s]c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\torch\nn\modules\module.py:1352: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "
100%|██████████| 938/938 [00:21<00:00, 43.25it/s] 


Epoch 2/10: train_loss=2.3016, test_loss=2.3015


100%|██████████| 938/938 [00:07<00:00, 129.82it/s]


Epoch 3/10: train_loss=2.3016, test_loss=2.3015


...

Epoch 8/10: train_loss=2.3015, test_loss=2.3010


100%|██████████| 938/938 [00:07<00:00, 125.62it/s]


Epoch 9/10: train_loss=2.3070, test_loss=2.3015


100%|██████████| 938/938 [00:07<00:00, 127.27it/s]


Epoch 10/10: train_loss=2.3016, test_loss=2.3013


In [6]:
# visualise training and testing data over time

epochs = np.arange(1, hyperparams["num_epochs"]+1)
train_fig = go.Figure()
for hidden_conv_layer, train_loss in train_losses.items():
    train_fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode='lines+markers', name=str(hidden_conv_layer)))

train_fig.update_layout(title="Training loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Hidden conv layers"
                  )
train_fig.show()

test_fig = go.Figure()
for hidden_conv_layer, test_loss in test_losses.items():
    test_fig.add_trace(go.Scatter(x=epochs, y=test_loss, mode='lines+markers', name=str(hidden_conv_layer)))

test_fig.update_layout(title="Test loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Hidden conv layers",
                  )
test_fig.show()

In [33]:
for hidden_conv_layer, model in models.items():
    t.save(model.state_dict(), f'../cnn_experiments/models/{hidden_conv_layer}-CNN.pth')

In [7]:
t.cuda.empty_cache()

In [8]:
# generate hessians
hessians = produce_hessians(models=models,
                            data_loader=test_loader,
                            num_batches=10,
                            criterion={"general" : criterion},
                            device=device)

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\torch\nn\modules\module.py:1352: UserWarning:

Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\torch\autograd\__init__.py:266: UserWarning:

Using backward() with create_graph=True will create a reference cycle between the parameter and its gradient which can cause a memory leak. We recommend using autograd.grad when creating the graph to avoid this. If you have to use this function, make sure to reset the .grad fields of your parameters to None after use to break the cycle and avoid the leak. (Triggered internally at ..\torch\csrc\autograd\engine.cpp:1182.)



In [9]:
# generate data and figures for eigenspectrum
figs, eigenspectrum_data = produce_hessian_eigenspectra(hessians, plot_type="log")

c:\Users\moosa\OneDrive\Documents\windows_dev\ngd_with_slt\notebooks\..\PyHessian\density_plot.py:62: ComplexWarning:

Casting complex values to real discards the imaginary part



In [10]:
for fig in figs:
    fig.show()

In [17]:
figs.append(train_fig)
figs.append(test_fig)

In [21]:
curr_time = datetime.now().strftime("%Y-%m-%d-%H-%M")
write_figs_to_html(figs, f"../cnn_experiments/cnn_hidden_layers_{curr_time}.html", title="Investigating effect of hidden conv layers on CNN Hessian eigenspectrum")